In [ ]:
#@title Language Selection / Выбор языка
#@markdown Choose your language / Выберите язык:
LANGUAGE = "English"  #@param ["English", "Russian"]

TEXTS = {
    "English": {
        "lang_selected": "Language set to English.",
        "mount_drive": "Mounting Google Drive...",
        "drive_mounted": "Google Drive mounted successfully.",
        "output_dir_created": "Output directory created: {}",
        "hardware_info": "Hardware detected: {} GPU, {} CPU cores, {:.1f} GB RAM",
        "no_gpu": "No GPU detected. Using CPU-only mode.",
        "gpu_detected": "GPU detected: {}",
        "choose_input": "Choose image input method:",
        "option_upload": "1 - Upload from PC",
        "option_drive": "2 - Load from Google Drive",
        "enter_choice": "Enter 1 or 2: ",
        "enter_path": "Enter the full path to the image in Google Drive: ",
        "file_not_found": "File not found: {}",
        "unsupported_format": "Unsupported format. Use PNG, JPG, or BMP.",
        "image_loaded": "Image loaded: {}x{} pixels",
        "choose_preset": "Choose a quality preset:",
        "preset_selected": "Preset selected: {}",
        "custom_prompt": "Enter custom value for {} (or press Enter to keep {}): ",
        "settings_summary": "Settings: stopAt={}, randomSamples={}, mutatedSamples={}, maxResolution={}",
        "starting": "Starting geometry generation...",
        "progress": "Shape {}/{} | Elapsed: {:.1f}s",
        "checkpoint_saved": "Checkpoint saved: {} ({} shapes)",
        "generation_complete": "Generation complete! {} shapes in {:.1f}s ({:.1f} shapes/s)",
        "saving_output": "Saving output to Google Drive...",
        "output_saved": "Output saved to: {}",
        "preview_title": "Generated Result Preview",
        "progress_bar_desc": "Generating shapes",
        "native_ready": "Native GPU generator ready - same speed as desktop!",
        "native_failed": "Native build unavailable. Will use Python fallback (slower).",
        "building": "Building native GPU generator...",
        "build_ok": "Build successful: {}",
        "build_fail": "Build failed.",
        "cloning": "Cloning generator source from GitHub...",
        "installing_deps": "Installing OpenCL runtime and build dependencies...",
    },
    "Russian": {
        "lang_selected": "Язык установлен: Русский.",
        "mount_drive": "Подключение Google Drive...",
        "drive_mounted": "Google Drive подключен успешно.",
        "output_dir_created": "Папка для вывода создана: {}",
        "hardware_info": "Оборудование: {} GPU, {} ядер CPU, {:.1f} ГБ RAM",
        "no_gpu": "GPU не обнаружен. Используется режим только CPU.",
        "gpu_detected": "GPU обнаружен: {}",
        "choose_input": "Выберите способ загрузки изображения:",
        "option_upload": "1 - Загрузить с компьютера",
        "option_drive": "2 - Загрузить из Google Drive",
        "enter_choice": "Введите 1 или 2: ",
        "enter_path": "Введите полный путь к изображению в Google Drive: ",
        "file_not_found": "Файл не найден: {}",
        "unsupported_format": "Неподдерживаемый формат. Используйте PNG, JPG или BMP.",
        "image_loaded": "Изображение загружено: {}x{} пикселей",
        "choose_preset": "Выберите пресет качества:",
        "preset_selected": "Выбран пресет: {}",
        "custom_prompt": "Введите значение для {} (или Enter для {}): ",
        "settings_summary": "Настройки: stopAt={}, randomSamples={}, mutatedSamples={}, maxResolution={}",
        "starting": "Начинается генерация геометрии...",
        "progress": "Фигура {}/{} | Время: {:.1f}с",
        "checkpoint_saved": "Контрольная точка сохранена: {} ({} фигур)",
        "generation_complete": "Генерация завершена! {} фигур за {:.1f}с ({:.1f} фигур/с)",
        "saving_output": "Сохранение результатов в Google Drive...",
        "output_saved": "Результат сохранен в: {}",
        "preview_title": "Предварительный просмотр результата",
        "progress_bar_desc": "Генерация фигур",
        "native_ready": "Нативный GPU генератор готов - скорость как на десктопе!",
        "native_failed": "Нативная сборка недоступна. Используется Python (медленнее).",
        "building": "Сборка нативного GPU генератора...",
        "build_ok": "Сборка успешна: {}",
        "build_fail": "Сборка не удалась.",
        "cloning": "Клонирование исходного кода генератора с GitHub...",
        "installing_deps": "Установка OpenCL и зависимостей сборки...",
    },
}

T = TEXTS[LANGUAGE]
print(T["lang_selected"])


In [ ]:
#@title Setup & Drive Mount / Настройка и подключение Drive
import os
import subprocess

from google.colab import drive

print(T["mount_drive"])
drive.mount("/content/drive")
print(T["drive_mounted"])

OUTPUT_DIR = "/content/drive/MyDrive/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(T["output_dir_created"].format(OUTPUT_DIR))

# Detect hardware
import multiprocessing
cpu_cores = multiprocessing.cpu_count()

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024**3)

GPU_AVAILABLE = False
GPU_NAME = "None"
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=10)
    if result.returncode == 0 and result.stdout.strip():
        GPU_NAME = result.stdout.strip().split("\n")[0]
        GPU_AVAILABLE = True
        print(T["gpu_detected"].format(GPU_NAME))
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

if not GPU_AVAILABLE:
    print(T["no_gpu"])

print(T["hardware_info"].format(GPU_NAME, cpu_cores, ram_gb))

# Ensure OpenCL vendor config exists for NVIDIA
os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
icd_path = "/etc/OpenCL/vendors/nvidia.icd"
if not os.path.exists(icd_path):
    with open(icd_path, "w") as f:
        f.write("libnvidia-opencl.so.1\n")
    print("  OpenCL vendor ICD configured for NVIDIA.")


In [ ]:
#@title Image Input / Загрузка изображения
from PIL import Image
from IPython.display import display

SUPPORTED_FORMATS = (".png", ".jpg", ".jpeg", ".bmp")

print(T["choose_input"])
print(T["option_upload"])
print(T["option_drive"])
choice = input(T["enter_choice"]).strip()

if choice == "1":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    filename = list(uploaded.keys())[0]
    image_path = os.path.join("/content", filename)
    with open(image_path, "wb") as f:
        f.write(uploaded[filename])
else:
    image_path = input(T["enter_path"]).strip()
    if not os.path.isfile(image_path):
        raise FileNotFoundError(T["file_not_found"].format(image_path))

ext = os.path.splitext(image_path)[1].lower()
if ext not in SUPPORTED_FORMATS:
    raise ValueError(T["unsupported_format"])

target_image = Image.open(image_path).convert("RGB")
img_width, img_height = target_image.size
print(T["image_loaded"].format(img_width, img_height))
display(target_image)


In [ ]:
#@title Settings / Preset Selection / Настройки
PRESETS = {
    "1": {
        "name": "Extremely Fast / Экстремально быстро",
        "stopAt": 500,
        "randomSamples": 30000,
        "mutatedSamples": 1000,
        "maxResolution": 600,
        "posterizeLevels": 10,
        "errorGridSize": 64,
        "maxThreads": 0,
        "forceOpaqueShapes": "true",
        "saveEvery": 100,
    },
    "2": {
        "name": "Fast / Быстро",
        "stopAt": 1000,
        "randomSamples": 60000,
        "mutatedSamples": 2000,
        "maxResolution": 900,
        "posterizeLevels": 10,
        "errorGridSize": 64,
        "maxThreads": 0,
        "forceOpaqueShapes": "true",
        "saveEvery": 200,
    },
    "3": {
        "name": "Balanced / Баланс",
        "stopAt": 1800,
        "randomSamples": 120000,
        "mutatedSamples": 5000,
        "maxResolution": 1400,
        "posterizeLevels": 20,
        "errorGridSize": 64,
        "maxThreads": 0,
        "forceOpaqueShapes": "true",
        "saveEvery": 360,
    },
    "4": {
        "name": "Slow / Медленно",
        "stopAt": 2500,
        "randomSamples": 220000,
        "mutatedSamples": 8000,
        "maxResolution": 2000,
        "posterizeLevels": 50,
        "errorGridSize": 64,
        "maxThreads": 0,
        "forceOpaqueShapes": "true",
        "saveEvery": 500,
    },
    "5": {
        "name": "Super Slow / Очень медленно",
        "stopAt": 3000,
        "randomSamples": 350000,
        "mutatedSamples": 12000,
        "maxResolution": 2400,
        "posterizeLevels": 100,
        "errorGridSize": 64,
        "maxThreads": 0,
        "forceOpaqueShapes": "true",
        "saveEvery": 600,
    },
}

print(T["choose_preset"])
for key, preset in PRESETS.items():
    print(f"  {key}. {preset['name']} (stopAt={preset['stopAt']}, samples={preset['randomSamples']})")

preset_choice = input(T["enter_choice"]).strip()
if preset_choice not in PRESETS:
    preset_choice = "3"

settings = dict(PRESETS[preset_choice])
print(T["preset_selected"].format(settings["name"]))

# Allow custom overrides
for param in ["stopAt", "randomSamples", "mutatedSamples", "maxResolution"]:
    val = input(T["custom_prompt"].format(param, settings[param])).strip()
    if val:
        settings[param] = int(val)

# Recalculate saveEvery based on stopAt
settings["saveEvery"] = max(1, round(settings["stopAt"] / 5))

print(T["settings_summary"].format(
    settings["stopAt"], settings["randomSamples"],
    settings["mutatedSamples"], settings["maxResolution"]
))
print(f"  saveEvery={settings['saveEvery']}, errorGridSize={settings['errorGridSize']}, "
      f"maxThreads={settings['maxThreads']}, forceOpaqueShapes={settings['forceOpaqueShapes']}")


In [ ]:
#@title Install & Build Native Generator / Установка нативного генератора
import subprocess, os

print(T["installing_deps"])
cmds = [
    "apt-get update -qq 2>/dev/null",
    "apt-get install -y -qq ocl-icd-opencl-dev opencl-headers clinfo 2>/dev/null",
    "mkdir -p /etc/OpenCL/vendors",
    "echo 'libnvidia-opencl.so.1' > /etc/OpenCL/vendors/nvidia.icd",
]
for cmd in cmds:
    subprocess.run(cmd, shell=True, capture_output=True)

# Verify OpenCL
r = subprocess.run(["clinfo", "--list"], capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print(f"  OpenCL devices:\n{r.stdout.strip()}")
else:
    print("  Warning: clinfo did not find devices, generator may still work via CUDA-OpenCL bridge")

# Check Go
r = subprocess.run(["go", "version"], capture_output=True, text=True)
if r.returncode != 0:
    print("Installing Go...")
    subprocess.run("apt-get install -y -qq golang-go 2>/dev/null", shell=True, capture_output=True)

r = subprocess.run(["go", "version"], capture_output=True, text=True)
print(f"  {r.stdout.strip()}")

# Clone generator
GENERATOR_DIR = "/content/geometrize-gpu"
if not os.path.exists(GENERATOR_DIR):
    print(f"\n{T['cloning']}")
    r = subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/zjl88858/forza-painter-geometrize-gpu.git", GENERATOR_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print(f"  Clone failed: {r.stderr[:300]}")
        GENERATOR_DIR = None
    else:
        print("  Cloned successfully")
else:
    print(f"\n  Source already at {GENERATOR_DIR}")

# Build
GENERATOR_BIN = None
if GENERATOR_DIR:
    print(f"\n{T['building']}")
    env = os.environ.copy()
    env["CGO_ENABLED"] = "1"
    r = subprocess.run(
        ["go", "build", "-o", "/content/geometrize-bin", "."],
        capture_output=True, text=True, cwd=GENERATOR_DIR, env=env, timeout=300
    )
    if r.returncode == 0:
        GENERATOR_BIN = "/content/geometrize-bin"
        os.chmod(GENERATOR_BIN, 0o755)
        print(f"  {T['build_ok'].format(GENERATOR_BIN)}")
        print(f"  {T['native_ready']}")
    else:
        print(f"  {T['build_fail']}")
        print(f"  {r.stderr[:500]}")
        GENERATOR_BIN = None

if not GENERATOR_BIN:
    print(f"\n  {T['native_failed']}")


In [ ]:
#@title Run Generation / Запуск генерации
import json, time, shutil, sys, os, subprocess
from pathlib import Path
from PIL import Image
from IPython.display import display
import numpy as np

# Prepare image
max_res = settings["maxResolution"]
img = Image.open(image_path).convert("RGB")
w, h = img.size
if max(w, h) > max_res:
    scale = max_res / max(w, h)
    img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
processed_path = os.path.join(WORK_DIR, "input.png")
img.save(processed_path)
proc_w, proc_h = img.size
print(T["starting"])
print(f"  Image: {proc_w}x{proc_h}")

if GENERATOR_BIN:
    # === NATIVE GO+OPENCL PATH ===
    # Write settings INI
    settings_path = os.path.join(WORK_DIR, "settings.ini")
    with open(settings_path, "w") as f:
        f.write("description = Colab settings\n")
        for key in ["maxResolution", "maxThreads", "randomSamples", "mutatedSamples",
                    "stopAt", "saveEvery", "posterizeLevels", "errorGridSize", "forceOpaqueShapes"]:
            if key in settings:
                f.write(f"{key} = {settings[key]}\n")

    output_base = os.path.join(WORK_DIR, "output")
    preview_path = os.path.join(WORK_DIR, "preview.png")

    cmd = [GENERATOR_BIN, processed_path, "-settings", settings_path, "-output", output_base, "-preview", preview_path]
    print(f"  Running: {' '.join(cmd)}\n")

    start_time = time.time()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in process.stdout:
        line = line.rstrip()
        if line:
            print(line)
        sys.stdout.flush()

    process.wait()
    elapsed = time.time() - start_time
    print(f"\n  Done in {elapsed:.1f}s (exit: {process.returncode})")

    # Find and copy outputs
    output_jsons = sorted(Path(WORK_DIR).glob("output*.json"), key=lambda p: p.stat().st_mtime)
    if not output_jsons:
        output_jsons = sorted(Path(WORK_DIR).glob("*.json"), key=lambda p: p.stat().st_mtime)

    image_basename = os.path.splitext(os.path.basename(image_path))[0]
    if output_jsons:
        for jp in output_jsons:
            shutil.copy2(str(jp), os.path.join(OUTPUT_DIR, jp.name))
        final = output_jsons[-1]
        drive_path = os.path.join(OUTPUT_DIR, f"{image_basename}_geometry.json")
        shutil.copy2(str(final), drive_path)
        print(T["output_saved"].format(drive_path))

    if os.path.exists(preview_path):
        print(f"\n{T['preview_title']}")
        display(Image.open(preview_path))
        shutil.copy2(preview_path, os.path.join(OUTPUT_DIR, f"{image_basename}_preview.png"))

else:
    # === PYTHON FALLBACK PATH ===
    from tqdm.notebook import tqdm

    target_arr = np.array(img, dtype=np.uint8)
    img_h, img_w = target_arr.shape[:2]
    avg_color = target_arr.mean(axis=(0, 1)).astype(np.uint8)
    canvas = np.full_like(target_arr, avg_color)
    RECT, ELLIPSE = 1, 16

    generated_shapes = [{"type": RECT, "data": [0, 0, img_w, img_h],
                         "color": [int(avg_color[0]), int(avg_color[1]), int(avg_color[2]), 255], "score": 0}]
    rng = np.random.default_rng(42)
    start_time = time.time()
    stop_at = settings["stopAt"]
    random_samples = min(settings["randomSamples"], 20000)  # Cap for Python
    mutated_samples = min(settings["mutatedSamples"], 1000)
    posterize_levels = settings["posterizeLevels"]

    def mask_np(cx, cy, hw, hh, rot, st, ih, iw):
        xn, xx, yn, yx = max(0, cx - hw), min(iw, cx + hw), max(0, cy - hh), min(ih, cy + hh)
        if xn >= xx or yn >= yx:
            return np.array([]), np.array([])
        ys, xs = np.mgrid[yn:yx, xn:xx]
        yf, xf = ys.ravel(), xs.ravel()
        dx, dy = xf - cx, yf - cy
        if rot != 0:
            r = rot * np.pi / 180
            c, s = np.cos(r), np.sin(r)
            rx, ry = dx * c + dy * s, -dx * s + dy * c
        else:
            rx, ry = dx.astype(float), dy.astype(float)
        if st == 1:
            m = (np.abs(rx) <= hw) & (np.abs(ry) <= hh)
        else:
            m = (rx * rx / (max(hw, 1)**2) + ry * ry / (max(hh, 1)**2)) <= 1.0 if hw > 0 and hh > 0 else np.zeros(len(yf), bool)
        return yf[m], xf[m]

    def post(r, g, b, l):
        if l <= 1:
            return int(r), int(g), int(b)
        s = 255.0 / (l - 1)
        return (max(0, min(255, int(round(round(r / s) * s)))),
                max(0, min(255, int(round(round(g / s) * s)))),
                max(0, min(255, int(round(round(b / s) * s)))))

    pbar = tqdm(range(1, stop_at + 1), desc=T["progress_bar_desc"], unit="shape", mininterval=0, dynamic_ncols=True)
    for si in pbar:
        bs, bsh, bc = -1e30, None, (128, 128, 128)
        for _ in range(random_samples):
            st = rng.choice([RECT, ELLIPSE])
            cx = int(rng.integers(0, img_w))
            cy = int(rng.integers(0, img_h))
            hw = int(rng.integers(2, max(4, img_w // 4)))
            hh = int(rng.integers(2, max(4, img_h // 4)))
            rot = int(rng.integers(0, 360))
            ys, xs = mask_np(cx, cy, hw, hh, rot, st, img_h, img_w)
            if len(ys) == 0:
                continue
            tp = target_arr[ys, xs]
            r, g, b = post(tp[:, 0].mean(), tp[:, 1].mean(), tp[:, 2].mean(), posterize_levels)
            cp_ = canvas[ys, xs].astype(float)
            tf = tp.astype(float)
            sc = float(np.sum((cp_ - tf)**2) - np.sum((np.array([r, g, b], dtype=float) - tf)**2))
            if sc > bs:
                bs, bsh, bc = sc, (st, cx, cy, hw, hh, rot), (r, g, b)
        if bsh is None:
            continue
        st, cx, cy, hw, hh, rot = bsh
        cs, cc = bs, bc
        for _ in range(mutated_samples):
            mt = int(rng.integers(0, 4))
            mx, my, mw, mh, mr = cx, cy, hw, hh, rot
            if mt == 0:
                mx = max(0, min(img_w - 1, cx + int(rng.integers(-20, 21))))
                my = max(0, min(img_h - 1, cy + int(rng.integers(-20, 21))))
            elif mt == 1:
                mw = max(2, min(img_w // 2, hw + int(rng.integers(-10, 11))))
            elif mt == 2:
                mh = max(2, min(img_h // 2, hh + int(rng.integers(-10, 11))))
            else:
                mr = (rot + int(rng.integers(-30, 31))) % 360
            ys, xs = mask_np(mx, my, mw, mh, mr, st, img_h, img_w)
            if len(ys) == 0:
                continue
            tp = target_arr[ys, xs]
            r, g, b = post(tp[:, 0].mean(), tp[:, 1].mean(), tp[:, 2].mean(), posterize_levels)
            sc = float(np.sum((canvas[ys, xs].astype(float) - tp.astype(float))**2) -
                       np.sum((np.array([r, g, b], dtype=float) - tp.astype(float))**2))
            if sc > cs:
                cx, cy, hw, hh, rot = mx, my, mw, mh, mr
                cs, cc = sc, (r, g, b)
        fr, fg, fb = cc
        ys, xs = mask_np(cx, cy, hw, hh, rot, st, img_h, img_w)
        if len(ys) > 0:
            canvas[ys, xs] = [fr, fg, fb]
        sd = [cx, cy, hw * 2, hh * 2] if st == RECT else [cx, cy, hw, hh, rot]
        generated_shapes.append({"type": st, "data": sd, "color": [fr, fg, fb, 255],
                                 "score": round(cs / (img_w * img_h * 3 * 255 * 255), 6)})
        el = time.time() - start_time
        sp = si / el if el > 0 else 0
        pbar.set_postfix(speed=f"{sp:.1f}/s")

    image_basename = os.path.splitext(os.path.basename(image_path))[0]
    op = os.path.join(OUTPUT_DIR, f"{image_basename}_geometry.json")
    with open(op, "w") as f:
        json.dump({"shapes": generated_shapes}, f, indent=2)
    print(T["output_saved"].format(op))
    print(f"\n{T['preview_title']}")
    from PIL import Image as PI
    pv = PI.fromarray(canvas)
    display(pv)
    pv.save(os.path.join(OUTPUT_DIR, f"{image_basename}_preview.png"))
    print(T["generation_complete"].format(stop_at, time.time() - start_time,
                                          stop_at / (time.time() - start_time)))


In [ ]:
#@title Output Summary / Итоги
import os

print("=" * 50)
print(f"Output folder: {OUTPUT_DIR}")
print("=" * 50)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        print(f"  {f} ({os.path.getsize(fp)/1024:.1f} KB)")
print(f"\nJSON files can be imported into FH6 via the desktop app Import function.")
